In [1]:

import os
import json
import random
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torch.optim import AdamW

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [2]:

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cpu")

MODEL_NAME = "answerdotai/ModernBERT-base"

MAX_LENGTH = 512
BATCH_SIZE = 2
EPOCHS = 2
LEARNING_RATE = 2e-4

DATA_PATH = Path("../Dataset")
MODEL_PATH = Path("../Models/modernbert")
RESULT_PATH = Path("../Results")

MODEL_PATH.mkdir(parents=True, exist_ok=True)
RESULT_PATH.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)
print("Model:", MODEL_NAME)
print("Maximum sequence length:", MAX_LENGTH)


Device: cpu
Model: answerdotai/ModernBERT-base
Maximum sequence length: 512


In [3]:

response_df = pd.read_json(
    DATA_PATH / "response.jsonl",
    lines=True
)

source_df = pd.read_json(
    DATA_PATH / "source_info.jsonl",
    lines=True
)

combined_df = response_df.merge(
    source_df,
    on="source_id",
    how="left"
)

combined_df["target"] = combined_df["labels"].apply(
    lambda x: 1 if isinstance(x, list) and len(x) > 0 else 0
)

model_df = combined_df[
    combined_df["quality"] == "good"
].copy()

print("Dataset shape:", model_df.shape)
print("\nClass distribution:")
print(model_df["target"].value_counts())

print("\nSplit distribution:")
print(model_df["split"].value_counts())

Dataset shape: (17617, 13)

Class distribution:
target
0    9976
1    7641
Name: count, dtype: int64

Split distribution:
split
train    14942
test      2675
Name: count, dtype: int64


In [4]:

original_train = model_df[
    model_df["split"] == "train"
].copy()

original_test = model_df[
    model_df["split"] == "test"
].copy()

# Select training examples
train_df, remaining_train = train_test_split(
    original_train,
    train_size=200,
    random_state=SEED,
    stratify=original_train["target"]
)

# Select validation examples from remaining training data
val_df, _ = train_test_split(
    remaining_train,
    train_size=50,
    random_state=SEED,
    stratify=remaining_train["target"]
)

# Select test examples from the official test split
test_df, _ = train_test_split(
    original_test,
    train_size=100,
    random_state=SEED,
    stratify=original_test["target"]
)

print("Training examples:", len(train_df))
print("Validation examples:", len(val_df))
print("Testing examples:", len(test_df))

print("\nTraining class distribution:")
print(train_df["target"].value_counts())


Training examples: 200
Validation examples: 50
Testing examples: 100

Training class distribution:
target
0    110
1     90
Name: count, dtype: int64


In [5]:

def convert_to_text(value):

    if isinstance(value, str):
        return value

    if isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)

    if value is None:
        return ""

    if isinstance(value, float) and np.isnan(value):
        return ""

    return str(value)


for df in [train_df, val_df, test_df]:

    df["source_info"] = df["source_info"].apply(convert_to_text)
    df["response"] = df["response"].apply(convert_to_text)

    df["source_info"] = df["source_info"].str.strip()
    df["response"] = df["response"].str.strip()


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully!")

Tokenizer loaded successfully!


In [6]:

class RAGDataset(Dataset):

    def __init__(self, dataframe):

        self.sources = dataframe["source_info"].tolist()
        self.responses = dataframe["response"].tolist()
        self.labels = dataframe["target"].astype(int).tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        encoding = tokenizer(
            self.sources[idx],
            text_pair=self.responses[idx],
            truncation=True,
            max_length=MAX_LENGTH
        )

        return {
            "input_ids": torch.tensor(
                encoding["input_ids"],
                dtype=torch.long
            ),
            "attention_mask": torch.tensor(
                encoding["attention_mask"],
                dtype=torch.long
            ),
            "labels": torch.tensor(
                self.labels[idx],
                dtype=torch.long
            )
        }


def collate_batch(batch):

    input_ids = pad_sequence(
        [item["input_ids"] for item in batch],
        batch_first=True,
        padding_value=tokenizer.pad_token_id
    )

    attention_mask = pad_sequence(
        [item["attention_mask"] for item in batch],
        batch_first=True,
        padding_value=0
    )

    labels = torch.stack(
        [item["labels"] for item in batch]
    )

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }


train_dataset = RAGDataset(train_df)
val_dataset = RAGDataset(val_df)
test_dataset = RAGDataset(test_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch
)

print("All datasets and loaders created successfully!")


All datasets and loaders created successfully!


In [7]:

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model.to(DEVICE)

# Freeze pretrained encoder
for param in model.base_model.parameters():
    param.requires_grad = False

# Keep classification head trainable
for param in model.classifier.parameters():
    param.requires_grad = True

optimizer = AdamW(
    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),
    lr=LEARNING_RATE
)

criterion = torch.nn.CrossEntropyLoss()

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("ModernBERT loaded successfully!")
print("Trainable parameters:", trainable_params)


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ModernBERT loaded successfully!
Trainable parameters: 592130


In [8]:

training_history = []

for epoch in range(EPOCHS):

    model.train()
    total_loss = 0

    print(f"\nEpoch {epoch + 1}/{EPOCHS}")

    for step, batch in enumerate(train_loader):

        batch = {
            key: value.to(DEVICE)
            for key, value in batch.items()
        }

        optimizer.zero_grad()

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )

        loss = criterion(
            outputs.logits,
            batch["labels"]
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if (step + 1) % 20 == 0:
            print(
                f"Step {step + 1}/{len(train_loader)} "
                f"| Loss: {loss.item():.4f}"
            )

    average_loss = total_loss / len(train_loader)

    training_history.append({
        "epoch": epoch + 1,
        "training_loss": average_loss
    })

    print("Average training loss:", round(average_loss, 4))

print("\nTraining completed!")



Epoch 1/2
Step 20/100 | Loss: 0.8286
Step 40/100 | Loss: 0.3278
Step 60/100 | Loss: 0.5949
Step 80/100 | Loss: 0.3675
Step 100/100 | Loss: 1.4585
Average training loss: 0.7338

Epoch 2/2
Step 20/100 | Loss: 0.3197
Step 40/100 | Loss: 0.6911
Step 60/100 | Loss: 1.0685
Step 80/100 | Loss: 0.4028
Step 100/100 | Loss: 0.3675
Average training loss: 0.5937

Training completed!


In [10]:

def evaluate_model(data_loader):

    model.eval()

    all_predictions = []
    all_labels = []

    with torch.no_grad():

        for batch in data_loader:

            outputs = model(
                input_ids=batch["input_ids"].to(DEVICE),
                attention_mask=batch["attention_mask"].to(DEVICE)
            )

            predictions = torch.argmax(
                outputs.logits,
                dim=1
            ).cpu().numpy()

            all_predictions.extend(predictions)
            all_labels.extend(batch["labels"].numpy())

    return np.array(all_labels), np.array(all_predictions)


print("Evaluation function ready!")


Evaluation function ready!


In [11]:
val_labels, val_predictions = evaluate_model(val_loader)

print("VALIDATION RESULTS")

print("Accuracy:",
      accuracy_score(val_labels, val_predictions))

print("Hallucination Precision:",
      precision_score(val_labels, val_predictions, zero_division=0))

print("Hallucination Recall:",
      recall_score(val_labels, val_predictions, zero_division=0))

print("Hallucination F1:",
      f1_score(val_labels, val_predictions, zero_division=0))

VALIDATION RESULTS
Accuracy: 0.68
Hallucination Precision: 0.7142857142857143
Hallucination Recall: 0.45454545454545453
Hallucination F1: 0.5555555555555556


In [12]:

test_labels, test_predictions = evaluate_model(test_loader)

print("FINAL TEST RESULTS\n")

print(classification_report(
    test_labels,
    test_predictions,
    labels=[0, 1],
    target_names=["Supported", "Hallucinated"],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(
    test_labels,
    test_predictions,
    labels=[0, 1]
))


FINAL TEST RESULTS

              precision    recall  f1-score   support

   Supported       0.74      0.78      0.76        65
Hallucinated       0.55      0.49      0.52        35

    accuracy                           0.68       100
   macro avg       0.64      0.64      0.64       100
weighted avg       0.67      0.68      0.68       100

Confusion Matrix:
[[51 14]
 [18 17]]


In [13]:

# Save model and tokenizer
model.save_pretrained(MODEL_PATH)
tokenizer.save_pretrained(MODEL_PATH)

# Save final metrics
results = pd.DataFrame([{
    "Model": "ModernBERT-base",
    "Training_Samples": len(train_df),
    "Validation_Samples": len(val_df),
    "Testing_Samples": len(test_df),
    "Accuracy": accuracy_score(test_labels, test_predictions),
    "Hallucination_Precision": precision_score(
        test_labels, test_predictions, zero_division=0
    ),
    "Hallucination_Recall": recall_score(
        test_labels, test_predictions, zero_division=0
    ),
    "Hallucination_F1": f1_score(
        test_labels, test_predictions, zero_division=0
    )
}])

results.to_csv(
    RESULT_PATH / "modernbert_results.csv",
    index=False
)

pd.DataFrame(training_history).to_csv(
    RESULT_PATH / "modernbert_training_history.csv",
    index=False
)

print("Model and results saved successfully!")
display(results)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and results saved successfully!


,Model,Training_Samples,Validation_Samples,Testing_Samples,Accuracy,Hallucination_Precision,Hallucination_Recall,Hallucination_F1
0,ModernBERT-base,200,50,100,0.68,0.548387,0.485714,0.515152


In [1]:

import os
import re
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

# Locate project root automatically
current_dir = Path.cwd()

if (current_dir / "Dataset").exists():
    project_root = current_dir
elif (current_dir.parent / "Dataset").exists():
    project_root = current_dir.parent
else:
    raise FileNotFoundError("Cannot locate the Dataset folder.")

# Load RAGTruth files
response_df = pd.read_json(
    project_root / "Dataset" / "response.jsonl",
    lines=True
)

source_df = pd.read_json(
    project_root / "Dataset" / "source_info.jsonl",
    lines=True
)

# Merge datasets
combined_df = response_df.merge(
    source_df,
    on="source_id",
    how="left"
)

# Create binary labels
combined_df["target"] = combined_df["labels"].apply(
    lambda x: 1 if len(x) > 0 else 0
)

# Retain good-quality responses
model_df = combined_df[
    combined_df["quality"] == "good"
].copy()

# Light text cleaning
def clean_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

model_df["source_info"] = model_df["source_info"].apply(clean_text)
model_df["response"] = model_df["response"].apply(clean_text)

# Separate original train and test sets
train_pool = model_df[
    model_df["split"] == "train"
].copy()

test_df = model_df[
    model_df["split"] == "test"
].copy()

# Create validation set from training data
train_df, val_df = train_test_split(
    train_pool,
    test_size=0.10,
    random_state=42,
    stratify=train_pool["target"]
)

# Reset indexes
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("DATA PREPARATION COMPLETE")
print("Project folder:", project_root)
print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Testing:", len(test_df))
print("\nTraining labels:")
print(train_df["target"].value_counts())


DATA PREPARATION COMPLETE
Project folder: c:\Users\Yashi\AI Projects\RAGGuard
Training: 13447
Validation: 1495
Testing: 2675

Training labels:
target
0    7419
1    6028
Name: count, dtype: int64


In [2]:

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

MODEL_NAME = "answerdotai/ModernBERT-base"
MAX_LENGTH = 1024
BATCH_SIZE = 8

# Load tokenizer
modern_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Dataset class
class RAGTruthDataset(Dataset):
    def __init__(self, dataframe):
        self.sources = dataframe["source_info"].tolist()
        self.responses = dataframe["response"].tolist()
        self.labels = dataframe["target"].tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        encoded = modern_tokenizer(
            self.sources[index],
            self.responses[index],
            truncation=True,
            max_length=MAX_LENGTH,
            padding="max_length",
            return_tensors="pt"
        )

        item = {
            key: value.squeeze(0)
            for key, value in encoded.items()
        }
        item["labels"] = torch.tensor(
            self.labels[index],
            dtype=torch.long
        )
        return item

# Create datasets
train_dataset = RAGTruthDataset(train_df)
val_dataset = RAGTruthDataset(val_df)
test_dataset = RAGTruthDataset(test_df)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Tokenization pipeline ready!")
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))


c:\Users\Yashi\tf_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tokenization pipeline ready!
Training batches: 1681
Validation batches: 187
Testing batches: 335


In [10]:

import torch
import torch.nn as nn
from transformers import AutoModel

DEVICE = torch.device("cpu")

# Load pretrained ModernBERT encoder
modern_encoder = AutoModel.from_pretrained(MODEL_NAME)

# Freeze pretrained encoder
for param in modern_encoder.parameters():
    param.requires_grad = False

# Classification model
class ModernBERTClassifier(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(
            encoder.config.hidden_size, 2
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Use first token representation
        pooled_output = outputs.last_hidden_state[:, 0, :]

        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)

        return logits

modern_model = ModernBERTClassifier(modern_encoder).to(DEVICE)

# Class weights to account for label imbalance
class_counts = train_df["target"].value_counts().sort_index()

weights = torch.tensor(
    [
        len(train_df) / (2 * class_counts[0]),
        len(train_df) / (2 * class_counts[1])
    ],
    dtype=torch.float32
).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=weights)

optimizer = torch.optim.AdamW(
    modern_model.classifier.parameters(),
    lr=2e-4
)

print("ModernBERT classifier ready!")
print("Device:", DEVICE)
print("Trainable parameters:",
      sum(p.numel() for p in modern_model.parameters() if p.requires_grad))
print("Class weights:", weights)


ModernBERT classifier ready!
Device: cpu
Trainable parameters: 1538
Class weights: tensor([0.9063, 1.1154])


In [11]:
# Freeze all model parameters
for param in modern_model.parameters():
    param.requires_grad = False

# Train only the classification head
for param in modern_model.classifier.parameters():
    param.requires_grad = True

# Verify
total = sum(p.numel() for p in modern_model.parameters())
trainable = sum(
    p.numel() for p in modern_model.parameters()
    if p.requires_grad
)

print("Total parameters:", f"{total:,}")
print("Trainable parameters:", f"{trainable:,}")
print("Frozen parameters:", f"{total-trainable:,}")

assert trainable == 1538, "Unexpected trainable parameter count!"

Total parameters: 149,015,810
Trainable parameters: 1,538
Frozen parameters: 149,014,272


In [12]:
import torch
import torch.nn as nn

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modern_model.to(DEVICE)

criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(
        [0.9063, 1.1154],
        dtype=torch.float32,
        device=DEVICE
    )
)

optimizer = torch.optim.AdamW(
    modern_model.classifier.parameters(),
    lr=1e-3,
    weight_decay=0.01
)

print("Feature extraction training configured!")

Feature extraction training configured!


In [13]:
import time
import torch

# Get one batch
batch = next(iter(train_loader))

# Separate labels from model inputs
model_inputs = {
    key: batch[key].to(DEVICE)
    for key in ["input_ids", "attention_mask"]
    if key in batch
}

modern_model.eval()

# Measure inference time
start_time = time.time()

with torch.no_grad():
    logits = modern_model(**model_inputs)

if hasattr(logits, "logits"):
    logits = logits.logits

elapsed = time.time() - start_time

print("One batch processed successfully!")
print("Batch size:", model_inputs["input_ids"].shape[0])
print("Output shape:", logits.shape)
print("Time taken:", round(elapsed, 2), "seconds")
print("Total training batches:", len(train_loader))

One batch processed successfully!
Batch size: 8
Output shape: torch.Size([8, 2])
Time taken: 18.96 seconds
Total training batches: 1681
